In [12]:
import pandas as pd
import numpy as np
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

In [13]:
df = pd.read_parquet('./data/taxi_data_preprocessed_missing_sampled.parquet')

In [14]:
mice_cols = [
    'passenger_count', 'RatecodeID', 'trip_distance',
    'fare_amount', 'PULocationID', 'DOLocationID',
    'extra', 'mta_tax', 'tolls_amount',
    'congestion_surcharge', 'Airport_fee', 'duration_min',
    'speed_mph'
]

df_mice_input = df[mice_cols].copy()
mice_imputer = IterativeImputer(max_iter=10, random_state=0)

df_mice_output = mice_imputer.fit_transform(df_mice_input)
df_imputed = pd.DataFrame(df_mice_output, columns=mice_cols)

df['passenger_count'] = df_imputed['passenger_count'].round().astype(int)
df['RatecodeID'] = df_imputed['RatecodeID'].round().astype(int)

print(df[['passenger_count', 'RatecodeID']].isnull().sum())

passenger_count    0
RatecodeID         0
dtype: int64


In [15]:
valid_rates = [1, 2, 3, 4, 5, 6, 99]
df['RatecodeID'] = df['RatecodeID'].apply(
    lambda x: min(valid_rates, key=lambda v: abs(v - x))
)

In [16]:
df.to_parquet('./data/taxi_data_preprocessed.parquet', index=False)

In [17]:
df = pd.read_parquet('./data/taxi_data_preprocessed.parquet')
df['passenger_count'] = df['passenger_count'].astype('int8')
df['pickup_hour'] = df['pickup_hour'].astype('int8')
df['day_of_week'] = df['day_of_week'].astype('int8')
df['is_yellow'] = df['is_yellow'].astype('int8')

tip_class_mapping = {'Low': 0, 'Middle': 1, 'High': 2}
df['tip_class'] = df['tip_class'].map(tip_class_mapping).astype('int8')
df.to_parquet('./data/taxi_data_preprocessed.parquet', index=False)

In [18]:
df = pd.read_parquet('./data/taxi_data_preprocessed.parquet')

In [19]:
df.columns

Index(['passenger_count', 'trip_distance', 'RatecodeID', 'PULocationID',
       'DOLocationID', 'fare_amount', 'extra', 'mta_tax', 'tolls_amount',
       'improvement_surcharge', 'congestion_surcharge', 'Airport_fee',
       'is_yellow', 'tip_class', 'pickup_hour', 'day_of_week', 'duration_min',
       'apparent_temperature', 'snowfall', 'precipitation', 'wind_speed_10m',
       'weather_code', 'speed_mph'],
      dtype='object')

In [20]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, TargetEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

target_col = 'tip_class'

numerical_features = [
    'passenger_count', 'trip_distance', 'fare_amount', 'extra', 'mta_tax',
    'tolls_amount', 'improvement_surcharge', 'congestion_surcharge',
    'Airport_fee', 'duration_min', 'apparent_temperature', 'snowfall',
    'precipitation', 'wind_speed_10m', 'speed_mph'
]

categorical_features_low_card = ['RatecodeID', 'weather_code', 'is_yellow']

categorical_features_high_card = ['PULocationID', 'DOLocationID']

def encode_cyclical(df, col, max_val):
    df[col + '_sin'] = np.sin(2 * np.pi * df[col]/max_val)
    df[col + '_cos'] = np.cos(2 * np.pi * df[col]/max_val)
    return df

df = encode_cyclical(df, 'pickup_hour', 23)
df = encode_cyclical(df, 'day_of_week', 6)

numerical_features.extend(['pickup_hour_sin', 'pickup_hour_cos', 'day_of_week_sin', 'day_of_week_cos'])

# 4. Split X and y
X = df.drop(columns=[target_col, 'pickup_hour', 'day_of_week'])
y = df[target_col]

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipeline_low = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

cat_pipeline_high = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('target_enc', TargetEncoder(target_type='auto', random_state=42))
])

preprocessor = ColumnTransformer([
    ('num', num_pipeline, numerical_features),
    ('cat_low', cat_pipeline_low, categorical_features_low_card),
    ('cat_high', cat_pipeline_high, categorical_features_high_card)
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train_processed = preprocessor.fit_transform(X_train, y_train)

X_test_processed = preprocessor.transform(X_test)

print("Data is ready for training!")
print(f"Processed Shape: {X_train_processed.shape}")

Data is ready for training!
Processed Shape: (1394141, 48)


In [21]:
import torch

X_train_tensor = torch.tensor(X_train_processed, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_processed, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train.values, dtype=torch.long)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.long)

In [22]:
# Save the tensors to a single file dictionary
torch.save({
    'X_train': X_train_tensor,
    'y_train': y_train_tensor,
    'X_test': X_test_tensor,
    'y_test': y_test_tensor,
}, './data/processed_taxi_data.pt')

print("Data saved to 'processed_taxi_data.pt'")

Data saved to 'processed_taxi_data.pt'
